In [ ]:
%matplotlib ipympl

In [ ]:
import functools
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
import scipy.interpolate as sci_interp
import scipy.optimize as sci_opt
import scipy.fft as sci_fft

jax.config.update("jax_enable_x64", True)

In [ ]:
def load_clean_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    data = np.array(pd.read_hdf(file_path))
    return data[:, 1:4], data[:, 4:]

file_path = "/Users/jozbee/work/eng/comp/data/clean_specific-forces-standard-road-v2.hdf"
acc_ref, omega_ref = load_clean_references(file_path)

acc_ref = jnp.clip(acc_ref, -1.0, 1.0)

## trig interp

In [ ]:
data_size = 90 * 200
dt = 0.005
data = acc_ref[:data_size, 0]

N = data.size
T = N * dt
K = 100

# Complex Fourier coefficients of the periodic interpolant
coeffs = sci_fft.fft(data) / N
coeffs = np.concatenate([coeffs[:K], coeffs[-K+1:]])
freqs = sci_fft.fftfreq(N, d=dt)   # cycles per second
freqs = np.concatenate([freqs[:K], freqs[-K+1:]])

def trig_interp(tq):
    tq = np.atleast_1d(tq)
    phase = np.exp(2j * np.pi * freqs[:, None] * tq[None, :])
    y = np.real(np.sum(coeffs[:, None] * phase, axis=0))
    return y if y.size > 1 else y.item()

# Evaluate on the original grid
ts = np.arange(N) * dt
data_fit = trig_interp(ts)

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))

ax.plot(ts, data, alpha=0.2)
ax.plot(ts, data_fit)
ax.set_ylim(-1.2, 1.2)
ax.grid()

## splines

In [ ]:
def mean_filt(data, window):
    data = np.array(data)
    assert len(data.shape) == 1
    indices = []
    filt = []
    idx = window
    while idx < data.size:
        indices.append(idx - window // 2)
        filt.append(np.mean(data[idx - window: idx]))
        idx += window
    return np.array(indices), np.array(filt)

In [ ]:
ts = np.arange(data.size) * 0.005
sampled_indices, sampled_data = mean_filt(data, 200)
sampled_ts = ts[sampled_indices]

# spline = sci_interp.PchipInterpolator(sampled_ts, sampled_data)
# spline = sci_interp.CubicSpline(sampled_ts, sampled_data)
spline = sci_interp.make_smoothing_spline(ts, data, lam=1e0)

fft_spline = sci_interp.make_smoothing_spline(ts, data_fit, lam=1e0)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
ax.plot(ts, data, label="data", alpha=0.1)
ax.plot(ts, spline(ts, nu=0), label="spline")
ax.plot(ts, fft_spline(ts, nu=0), label="fft_spline")
# ax.plot(ts, full_eval_spline[:, 0], label="spline")
ax.legend()
ax.grid()

## splrep

In [ ]:
k = 3
t = ts[::100][1:-1]
t = np.concatenate([
    np.flip(np.arange(1, k + 2) * -0.005),
    t,
    ts[-1] + np.arange(1, k + 2) * 0.005,
])
slerp = sci_interp.make_splrep(ts, data, k=k, s=1, t=t)
# ignore the warning

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
ax.plot(ts, data, label="data", alpha=0.2)
ax.plot(ts, slerp(ts, nu=0), label="slerp")
ax.legend()
ax.grid()

## $\alpha$ filter (also minimize derivative)

In [ ]:
@jax.jit
def alpha_filt(alpha, data):
    assert len(data.shape) == 1
    
    def scan_body(x0, x):
        x0 += alpha * (x - x0)
        return x0, x0

    _, res = jax.lax.scan(scan_body, data[0], data[1:])
    return jnp.concatenate([np.reshape(data[0], shape=(1,)), res])

@jax.jit
def alpha_cost(alpha, data):
    alpha = jnp.squeeze(alpha)
    af = alpha_filt(alpha, data)
    afp = jnp.diff(af) * 200.0
    return jnp.mean(jnp.square(af - data)) + jnp.mean(jnp.square(afp)) * 2e-2

alpha_cost_grad = jax.jit(jax.value_and_grad(alpha_cost))
alpha_grad = jax.jit(jax.grad(alpha_cost))

a_solver = optax.adam(learning_rate=0.01)
a_params = jnp.array(0.01)
a_opt_state = a_solver.init(a_params)
a_grad = alpha_grad(a_params, data)

In [ ]:
iter = 0
while jnp.linalg.norm(a_grad) > 1e-5 and iter < 1000:
    updates, a_opt_state = a_solver.update(a_grad, a_opt_state, a_params)
    a_params = optax.apply_updates(a_params, updates)
    a_grad = alpha_grad(a_params, data)
    iter += 1
iter

In [ ]:
# res = sci_opt.minimize(
#     fun=functools.partial(alpha_cost_grad, data=data),
#     x0=a_params,
#     bounds=[(0, 1)],
#     method="L-BFGS-B",
#     jac=True,
#     tol=1e-5,
# )
# a_params = float(np.squeeze(res.x))

In [ ]:
tmp = alpha_cost_grad(a_params, data)
float(a_params), float(tmp[0]), float(jnp.linalg.norm(tmp[1])), float(jnp.linalg.norm(a_grad))

In [ ]:
plot_range = [0, 90 * 200]
# plot_range = [1000 * 200, 1100 * 200]
plot_data = acc_ref[plot_range[0]: plot_range[1], 0]
alpha_data = alpha_filt(a_params, plot_data)

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(14, 7))
ax.plot(plot_data, label="plot_data", alpha=0.4)
ax.plot(alpha_data, label="alpha")
ax.legend()
ax.grid()

## $\alpha \beta$ filter

In [ ]:
dt = 0.005

@jax.jit
def ab_filt(ab, data):
    assert ab.shape == (2,)
    assert len(data.shape) == 1

    a, b = ab
    
    def scan_body(xv0, x):
        x0, v0 = xv0
        r = x - x0
        x0 += v0 * dt + a * r
        v0 += b / dt * r
        xv0 = jnp.array([x0, v0])
        return xv0, xv0

    _, res = jax.lax.scan(scan_body, jnp.array([data[0], 0.0]), data[1:])
    return jnp.concatenate([np.reshape(data[0], shape=(1,)), res[:, 0]])

@jax.jit
def ab_cost(ab, data_in, data_ref=None):
    if data_ref is None:
        data_ref = data_in
        data_refp = jnp.diff(jnp.zeros_like(data_ref)) * dt**-1
    else:
        data_refp = jnp.diff(data_ref) * dt**-1
    abf = ab_filt(ab, data_in)
    abfp = jnp.diff(abf) * dt**-1
    res = jnp.mean(jnp.square(abf - data_ref))
    res += jnp.mean(jnp.square(abfp - data_refp)) * 1e-2
    return res

ab_cost_grad = jax.jit(jax.value_and_grad(ab_cost))
ab_grad_fun = jax.jit(jax.grad(ab_cost))

ab_solver = optax.adam(learning_rate=0.001)
ab_params = jnp.array([0.0, 0.0001])
ab_opt_state = ab_solver.init(ab_params)
ab_grad = ab_grad_fun(ab_params, data)

In [ ]:
# iter = 0
# while jnp.linalg.norm(ab_grad) > 1e-5 and iter < 1000:
#     updates, ab_opt_state = ab_solver.update(ab_grad, ab_opt_state, ab_params)
#     ab_params = optax.apply_updates(ab_params, updates)
#     ab_grad = ab_grad_fun(ab_params, data)
#     iter += 1
# iter

In [ ]:
res = sci_opt.minimize(
    # fun=functools.partial(ab_cost_grad, data_in=data),
    fun=functools.partial(ab_cost_grad, data_in=data, data_ref=slerp(ts)),
    x0=ab_params,
    bounds=[(0, 1), (0, 1)],
    method="L-BFGS-B",
    jac=True,
    tol=1e-10,
)
ab_params = np.squeeze(res.x)

In [ ]:
ab_data = ab_filt(ab_params, data)

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
ax.plot(data, label="data", alpha=0.2)
# ax.plot(ts, slerp(ts), label="slerp")
ax.plot(ab_data, label="ab")
ax.legend()
ax.grid()